<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/07_sft_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation of Fine-tuned Sentiment Analysis Model
This notebook provides a comprehensive evaluation of the **Gemma 3 1B-IT** model after it has been fine-tuned for financial sentiment analysis with reasoning. 

### Why Multi-Dimensional Evaluation?
A simple accuracy score is often insufficient for Large Language Models. To truly understand the model's performance, we evaluate it across three critical dimensions:
1. **Sentiment Accuracy**: Does the predicted label (Positive/Neutral/Negative) match the ground truth?
2. **Tag Integrity (Instruction Following)**: Did the model correctly use the requested `<sentiment>` and `<reasoning>` tags? This is vital for parsing the output in automated pipelines.
3. **Reasoning Quality (LLM-as-a-judge)**: How insightful and accurate are the model's explanations? We use a larger model (**Qwen 2.5 7B**) to grade these explanations on a scale of 1-10.

### Hardware Verification

Before starting, we check the GPU status using `nvidia-smi`. 

| Metric | Description |
|---|---|
| `GPU-Util` | Percentage of GPU cores currently active. |
| `Memory-Usage` | Amount of VRAM occupied. Loading our model in 4-bit will typically use ~2-3GB. |
| `Perf` | Performance state (P0 is max performance). |

In [1]:
import platform

if platform.system() != "Darwin":
    # Check the GPU information
    !nvidia-smi

Tue Mar 24 07:48:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.142                Driver Version: 580.142        CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GB10                    On  |   0000000F:01:00.0 Off |                  N/A |
| N/A   42C    P0              9W /  N/A  | Not Supported          |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Environment Setup

We install and import the necessary libraries. We'll use `peft` to load our LoRA adapter and `sklearn` for calculating traditional classification metrics.

In [2]:
%%capture
# Install necessary libraries for model training and evaluation
import os

if "COLAB_" not in "".join(os.environ.keys()):
    pass
else:
    !pip install -U transformers trl accelerate bitsandbytes

In [3]:
# Suppress non-critical warnings to keep notebook output clean
import warnings

warnings.filterwarnings("ignore")

In [4]:
# Numeric and data utilities
import pandas as pd
import os
import gc
import re  # regular expressions for parsing model output tags
from tqdm import tqdm

# PyTorch
import torch
from torch.nn.utils.rnn import pad_sequence

# Hugging Face Transformers
import transformers
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,  # 4-bit quantization configuration
)

# Import the Gemma3-specific model class directly for 4-bit quantization support
from transformers.models.gemma3 import Gemma3ForCausalLM

# Hugging Face ecosystem
from huggingface_hub import login
from datasets import load_dataset

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt

## 1. Model Loading

We load our fine-tuned model. Note that we first load the **base model** in 4-bit quantization and then attach the **LoRA adapter**. This "PeftModel" approach is memory-efficient and allows us to switch between different fine-tuned versions easily.

In [5]:
def define_device():
    """Determines and returns the optimal PyTorch device based on hardware availability."""

    print(f"PyTorch version: {torch.__version__}", end=" -- ")

    # Prefer MPS (Metal Performance Shaders) on Apple Silicon Macs
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("using MPS device on macOS")
        return torch.device("mps")

    # Fall back to CUDA (NVIDIA GPU) or CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"using {device}")
    return device


device = define_device()
print(f"Device: {device}")

PyTorch version: 2.10.0+cu130 -- using cuda
Device: cuda


In [6]:
def init():
    """Sets environment variables, logs in to Hugging Face, and clears GPU memory."""
    os.environ["TOKENIZERS_PARALLELISM"] = (
        "false"  # avoid deadlocks in forked tokenizer processes
    )
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # pin to the first GPU

    token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token)
    else:
        try:
            from google.colab import userdata

            # Read the token stored in Colab's Secrets tab (🔑 icon)
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                login(token=hf_token, add_to_git_credential=True)
                print("Hugging Face login successful using Google Colab secrets!")
            else:
                print("Error: HF_TOKEN not found in Google Colab secrets or is empty.")
                print(
                    "Please ensure you have created a secret named 'HF_TOKEN' in the 'Secrets' tab (🔑) on the left sidebar."
                )
        except:
            print("HF_TOKEN not set. You might need to log in manually.")

    torch.cuda.empty_cache()  # release any stale GPU allocations from a previous run
    gc.collect()
    warnings.filterwarnings("ignore")


init()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
# Use the local path or Hub repo for your fine-tuned model
MODEL_PATH = "lmassaron/qlora4b-Gemma3-1B-finsent"
DATASET_ID = "lmassaron/FinancialPhraseBank_explained"

print("Loading test dataset...")
ds = load_dataset(DATASET_ID, split="test")
test_df = ds.to_pandas()

Loading test dataset...


In [8]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_PATH = "lmassaron/qlora4b-Gemma3-1B-finsent"

# 1. Read adapter config to get the base model id
peft_config = PeftConfig.from_pretrained(MODEL_PATH)
BASE_MODEL_ID = peft_config.base_model_name_or_path  # e.g. "google/gemma-3-1b"

# 2. Load the quantized base model
print(f"Loading model from {MODEL_PATH}...")
QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=QUANTIZATION_CONFIG,
    device_map="auto",
)

# 3. Attach the LoRA adapter on top
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model = model.eval()

Loading model from lmassaron/qlora4b-Gemma3-1B-finsent...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

## 2. Batch Inference

We run inference on the test set. 

Crucially, we use the same `INSTRUCTION` prompt that the model saw during fine-tuning. We capture the **raw output** and use regular expressions (`re`) to extract the contents of our custom tags.

In [9]:
INSTRUCTION = (
    "You are a financial analyst with expertise in equity markets and corporate finance.\n"
    "Analyze the following financial news headline and determine its market sentiment "
    "from an investor's perspective.\n\n"
    "Classify the sentiment as positive, neutral, or negative based on the likely "
    "impact on stock price, investor confidence, or financial performance.\n\n"
    "Respond using exactly these two tags:\n"
    "<sentiment>positive|neutral|negative</sentiment>\n"
    "<reasoning>brief financial explanation</reasoning>\n"
)


def extract_tag(text, tag):
    match = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None


results = []
batch_size = 16

for start in tqdm(range(0, len(test_df), batch_size), desc="Evaluating"):
    batch_df = test_df.iloc[start : start + batch_size]

    prompts = []
    for _, row in batch_df.iterrows():
        messages = [
            {
                "role": "user",
                "content": INSTRUCTION + f'\nHeadline: "{row["sentence"]}"\n',
            }
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )

    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

    for i, raw_output in enumerate(decoded):
        sentiment_pred = extract_tag(raw_output, "sentiment")
        reasoning_pred = extract_tag(raw_output, "reasoning")

        results.append(
            {
                "sentence": batch_df.iloc[i]["sentence"],
                "sentiment_true": batch_df.iloc[i]["sentiment"],
                "raw_output": raw_output,
                "sentiment_pred": sentiment_pred.lower() if sentiment_pred else "none",
                "reasoning_pred": reasoning_pred if reasoning_pred else "none",
                "sentiment_tag_present": sentiment_pred is not None,
                "reasoning_tag_present": reasoning_pred is not None,
            }
        )

results_df = pd.DataFrame(results)

Evaluating: 100%|██████████| 31/31 [01:52<00:00,  3.63s/it]


## 3. Metric 1: Sentiment Accuracy

We use `classification_report` to see how well the model performs for each category. 

**Key Metrics:**
- **Precision**: When the model predicts "Positive", how often is it actually positive?
- **Recall**: Out of all actual "Positive" headlines, how many did the model find?
- **F1-Score**: The harmonic mean of precision and recall.

In [10]:
accuracy = accuracy_score(results_df["sentiment_true"], results_df["sentiment_pred"])
print(f"Overall Sentiment Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(results_df["sentiment_true"], results_df["sentiment_pred"]))

Overall Sentiment Accuracy: 77.89%

Classification Report:
              precision    recall  f1-score   support

    negative       0.84      0.87      0.85        61
     neutral       0.87      0.80      0.83       287
        none       0.00      0.00      0.00         0
    positive       0.83      0.70      0.76       136

    accuracy                           0.78       484
   macro avg       0.64      0.59      0.61       484
weighted avg       0.86      0.78      0.82       484



## 4. Metric 2: Tag Integrity

This measures instruction following. If the model fails to output `<sentiment>` or `<reasoning>` tags, it will break downstream processing. A score of 100% is the goal for production reliability.

In [11]:
sentiment_tag_rate = results_df["sentiment_tag_present"].mean()
reasoning_tag_rate = results_df["reasoning_tag_present"].mean()
both_tags_rate = (
    results_df["sentiment_tag_present"] & results_df["reasoning_tag_present"]
).mean()

print(f"<sentiment> tag present: {sentiment_tag_rate:.2%}")
print(f"<reasoning> tag present: {reasoning_tag_rate:.2%}")
print(f"Both tags correctly formed: {both_tags_rate:.2%}")

<sentiment> tag present: 90.91%
<reasoning> tag present: 90.91%
Both tags correctly formed: 90.91%


## 5. Metric 3: Reasoning Quality (LLM-as-a-judge)

How do we know if the reasoning is actually "good"? We use a larger, more capable model (**Qwen 2.5 7B**) as an impartial judge. 

### Evaluation Rubric (1-10)
- **1-2**: Wrong, irrelevant, or missing reasoning.
- **5-6**: Accurate but generic; doesn't link headline to market outcomes.
- **9-10**: Precise, insightful, and concise; clearly justifies the sentiment with financial logic.

This step effectively "distills" the knowledge of a 7B model into a 1B model and then uses the 7B model to verify the quality of the distillation.

In [12]:
# Loading Judge Model
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading Judge Model: {JUDGE_MODEL_ID}...")
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    # quantization_config=QUANTIZATION_CONFIG,
    device_map="auto",
)

JUDGE_PROMPT = (
    "You are a master financial analyst judging the quality of reasoning for a sentiment classification.\n\n"
    'Headline: "{sentence}"\n'
    "Classified Sentiment: {sentiment}\n"
    'Provided Reasoning: "{reasoning}"\n\n'
    "Score the reasoning 1-10 across financial accuracy, depth of insight, and clarity:\n"
    "  1-2  Wrong, irrelevant, or missing.\n"
    "  3-4  Partially correct but shallow or off-topic.\n"
    "  5-6  Accurate but generic; no link to price, confidence, or performance.\n"
    "  7-8  Solid and relevant; connects the headline to a financial outcome.\n"
    "  9-10 Precise, insightful, and concise; clearly justifies the sentiment.\n\n"
    "Respond with a single number from 1 to 10. Do not provide any other text."
)

BATCH_SIZE = 8
scores = []
# eval_sample = results_df.sample(min(BATCH_SIZE * 20, len(results_df))).reset_index(drop=True)
rows = results_df.to_dict("records")  # Convert to dicts for faster access than iterrows

# Ensure padding token is set
if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token

for batch_start in tqdm(range(0, len(rows), BATCH_SIZE), desc="Judging Reasoning"):
    batch = rows[batch_start : batch_start + BATCH_SIZE]

    # 1. Prepare data and track indices
    valid_prompts = []
    valid_indices = []
    batch_scores = [1] * len(batch)  # Default to 1 for "none" reasoning cases

    for i, row in enumerate(batch):
        if row["reasoning_pred"].lower() != "none":
            prompt = JUDGE_PROMPT.format(
                sentence=row["sentence"],
                sentiment=row["sentiment_pred"],
                reasoning=row["reasoning_pred"],
            )
            # Use the chat template format (list of dicts)
            valid_prompts.append([{"role": "user", "content": prompt}])
            valid_indices.append(i)

    # 2. Batch Tokenization (Efficient)
    if not valid_prompts:
        # If the whole batch was "none", just extend and continue
        scores.extend(batch_scores)
        continue

    judge_tokenizer.padding_side = "left"  # Required for decoder-only generation
    inputs = judge_tokenizer.apply_chat_template(
        valid_prompts,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,
        return_tensors="pt",
        return_dict=True,
    ).to(judge_model.device)

    # 3. Inference
    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=judge_tokenizer.pad_token_id,
        )

    # 4. Decoding and Extraction
    prompt_len = inputs["input_ids"].shape[1]

    for local_idx, original_idx in enumerate(valid_indices):
        # Extract only the generated tokens
        generated_tokens = outputs[local_idx][prompt_len:]
        decoded = judge_tokenizer.decode(
            generated_tokens, skip_special_tokens=True
        ).strip()

        # Regex to find the first number 1-10
        match = re.search(r"\b([1-9]|10)\b", decoded)
        if match:
            batch_scores[original_idx] = int(match.group(1))
        else:
            batch_scores[original_idx] = None  # Or keep as 1 if you prefer

    scores.extend(batch_scores)

# Finalize
results_df["reasoning_score"] = scores
avg_score = results_df["reasoning_score"].dropna().mean()
print(f"\nAverage Reasoning Quality Score: {avg_score:.2f} / 10")

Loading Judge Model: Qwen/Qwen2.5-7B-Instruct...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Judging Reasoning: 100%|██████████| 61/61 [00:36<00:00,  1.66it/s]


Average Reasoning Quality Score: 6.45 / 10
